# Tests 2 & 4: MD Stability and Condensed-Phase Water

Colab-runnable driver for Test 2 (short-MD molecular stability, 4 small
molecules) and Test 4 (condensed-phase 168-water box). Separate from
`01_dimer_scan.ipynb` (Test 3) -- different environment stack (`md`, not
`ani-mace`/`uma`) and different scope.

**Read `RESULTS.md`'s "Test 2 and Test 4" section before running this** --
it documents every way this session's protocol deviates from Ranasinghe
et al. 2025's actual Test 2/4 setup (shortened production lengths, no
solute in Test 4, `inference_settings="turbo"` never having been
exercised anywhere but here, etc.). Those deviations are compute-budget
and scope decisions already made -- this notebook just runs them.

**Both tests are resumable.** If Colab disconnects, just re-run the same
cell -- `run_resumable_md` picks up from the last checkpoint.

**Use Google Drive for `MD_ROOT`** so results/checkpoints survive a
runtime recycle -- see the Drive-mount cell below. This is NOT optional
for Test 4 in particular: 175 ps of real MD on ~500 atoms will not finish
in one Colab session, and losing checkpoints on disconnect would waste
the compute-unit budget this work is explicitly scoped against.

## 1. Mount Google Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# All Test 2/4 checkpoints and results go here -- survives a runtime recycle.
os.environ['MLIP_AUDIT_MD_ROOT'] = '/content/drive/MyDrive/mlip-audit-md'
os.makedirs(os.environ['MLIP_AUDIT_MD_ROOT'], exist_ok=True)
print('MD_ROOT set to:', os.environ['MLIP_AUDIT_MD_ROOT'])

In [ ]:
REPO_URL = "<your-fork-or-repo-url>"
REPO_DIR = "/content/mlip-audit"

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

In [ ]:
!bash setup.sh md

## 2. HuggingFace login (only if setup.sh reported no cached token)

Required for UMA-S and both eSEN checkpoints. Make sure you've accepted
both gated model licenses on huggingface.co with this account.

In [ ]:
from huggingface_hub import get_token

if get_token() is None:
    from huggingface_hub import login
    login()  # interactive widget; do not pass token= here
else:
    print("Already logged in.")

## 3. Verify `inference_settings="turbo"` actually works here

This has NEVER been exercised successfully anywhere in this project --
only tested locally on a machine without a C++ compiler, where it fails.
Colab normally has gcc, so it should work, but VERIFY before trusting any
downstream result. If this cell errors, fall back to
`inference_settings="batch"` in `mlip_audit/models.py` (slower, but the
one mode validated so far in this project) and re-run.

In [ ]:
from mlip_audit.models import get_calc, prepare_atoms_for_model
from mlip_audit.molecules import build_molecule_from_smiles

atoms = build_molecule_from_smiles("CCO", seed=0)  # ethanol, cheap
calc = get_calc("uma-s-1p1", inference_settings="turbo")
atoms.calc = calc
prepare_atoms_for_model(atoms, "uma-s-1p1", charge=0, spin=1)
e = atoms.get_potential_energy()
print("turbo mode energy:", e, "eV -- if this printed with no error, turbo works here.")

## 4. Test 2: short-MD molecular stability

In [ ]:
# Full sweep: 4 molecules x 4 models x 2 seeds = 32 runs, each 100 ps.
# Resumable -- safe to re-run this exact cell after any disconnect.
# To run a subset (e.g. while iterating), pass --molecule/--model/--seed.
!python -m mlip_audit.test2_md_stability --verbose

In [ ]:
# Bond-length stability analysis across the full sweep.
from mlip_audit.config import MD_ROOT, TEST2_MOLECULES, TEST2_MODELS, TEST2_SEEDS
from mlip_audit.md_analysis import bond_length_trajectory_report

for molecule in TEST2_MOLECULES:
    for model in TEST2_MODELS:
        for seed in TEST2_SEEDS:
            traj_path = MD_ROOT / "test2" / molecule / model / f"seed{seed}" / "md.traj"
            if not traj_path.exists():
                print(f"{molecule}/{model}/seed{seed}: NOT RUN YET")
                continue
            report = bond_length_trajectory_report(traj_path)
            flag = "UNSTABLE" if report["any_unstable"] else "stable"
            print(f"{molecule}/{model}/seed{seed}: {report['n_frames']} frames, {flag}")
            if report["any_unstable"]:
                for b in report["bonds"]:
                    if b["flagged_unstable"]:
                        print(f"    bond {b['symbols']} (atoms {b['i']},{b['j']}): "
                              f"d0={b['d0']:.3f} A, max={b['d_max']:.3f} A, ratio={b['max_ratio']:.2f}x")

## 5. Test 4: condensed-phase water

In [ ]:
# Full sweep: 4 models, each 125 ps NVT + 50 ps NPT. Resumable per-phase.
# This is the most compute-intensive part of this notebook -- consider
# running one model at a time (--model uma-s-1p1, etc.) across multiple
# sessions if a single session's time/compute-unit budget is tight.
!python -m mlip_audit.test4_condensed_water --verbose

In [ ]:
# O-O radial distribution function from each model's NPT production phase.
import matplotlib.pyplot as plt
from mlip_audit.config import MD_ROOT, TEST4_MODELS
from mlip_audit.md_analysis import oo_radial_distribution_function

fig, ax = plt.subplots(figsize=(8, 6))
for model in TEST4_MODELS:
    npt_traj = MD_ROOT / "test4" / model / "npt" / "npt.traj"
    if not npt_traj.exists():
        print(f"{model}: NPT not run yet")
        continue
    rdf = oo_radial_distribution_function(npt_traj)
    ax.plot(rdf["r"], rdf["g_r"], label=f"{model} ({rdf['n_frames_used']} frames)")

ax.axvline(2.8, color="gray", linestyle="--", linewidth=1, label="experimental O-O peak (~2.8 A)")
ax.set_xlabel("r (A)")
ax.set_ylabel("g_OO(r)")
ax.set_title("Test 4: O-O radial distribution function")
ax.legend()
ax.grid(True, alpha=0.3)
fig.savefig(str(MD_ROOT / "test4" / "rdf_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Record results

Results already live on Google Drive (`MD_ROOT`), so there's no separate
download step needed here (unlike Test 3's notebook) -- but do copy the
key findings (bond-stability table, RDF plot, any instabilities) into
`RESULTS.md`'s "Test 2 and Test 4" section by hand once a run completes,
the same way every other result in this project has been documented.